# Create Question-Answer Pairs

In [9]:
import ast
import json
import os
import pandas as pd
import requests

from dotenv import load_dotenv
from IPython.display import Image
from pathlib import Path
from rapidfuzz import process, fuzz
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from typing import Dict, Any

In [10]:
# Load splits from CSVs
df_train = pd.read_csv('../data/train/df_train_with_iron.csv')
df_val = pd.read_csv('../data/val/df_val_with_iron.csv')
df_test = pd.read_csv('../data/test/df_test_with_iron.csv')

df_train.head(2)

,image_url,camera_or_phone_prob,food_prob,dish_name,food_type,ingredients,portion_size,nutritional_profile,cooking_method,sub_dt,image_name,"Iron, Fe","Calcium, Ca","Vitamin C, total ascorbic acid"
0,https://file.b18a.io/7832973280900104501_54585...,0.8,0.90,oysters,homemade food,['oysters'],{'oysters': '500g'},"{'fat_g': 5.0, 'protein_g': 20.0, 'calories_kc...",raw,20250710,7832973280900104501_545859_.jpeg,0.90,160.0,0.50
1,https://file.b18a.io/7835136777400102715_70587...,0.7,0.95,grilled steak,restaurant food,"['steak', 'broccoli', 'potato', 'tomato', 'sau...","{'steak': '250g', 'broccoli': '50g', 'potato':...","{'fat_g': 30.0, 'protein_g': 50.0, 'calories_k...",grilling,20250702,7835136777400102715_705873_.jpeg,5.85,102.0,36.35


In [11]:
def generate_qa_pairs(cur_img_data: Dict[str, Any]) -> list:
    # Generate question-answer pairs
    # Init pairs
    qa_pairs = []

    # Q1. What is the general name of this dish?
    # Example answer: oysters
    qa_pairs.append({
            "question": "What is the general name of this dish?",
            "answer": cur_img_data['dish_name'],
        })

    # Q2. What category of food is this?
    # Example answer: homemade food
    qa_pairs.append({
            "question": "What category of food is this?",
            "answer": cur_img_data['food_type'],
        })

    # Q3. What primary cooking method was used for this dish?
    # Example answer: raw
    qa_pairs.append({
            "question": "What primary cooking method was used for this dish?",
            "answer": cur_img_data['cooking_method'],
        })

    # Q4. What is the estimated weight of the ingredient '[ingredient_name]' in grams?
    # - Multiple Qs:
    #   - for each ingredient, weight in df['portion_size'].items()
    # Example answer: 500.0
    total_weight = 0
    for ingredient, weight in cur_img_data['portion_size'].items():
        if weight.endswith('kg'):
            weight = float(weight.replace('kg', '')) * 1000
        elif weight.endswith('g'):
            weight = float(weight.replace('g', ''))
        elif weight.endswith('ml'):
            weight = float(weight.replace('ml', ''))
        elif weight.endswith('mL'):
            weight = float(weight.replace('mL', ''))
        elif weight.endswith('L'):
            weight = float(weight.replace('L', '')) * 1000
        else:
            print('Invalid weight!', ingredient, weight)
            continue
        qa_pairs.append({
            "question": f"What is the estimated weight of the ingredient '{ingredient}' in grams?",
            "answer": weight,
        })    
        # Also calculate total weight for Q6
        total_weight += weight
    # print(f'{total_weight=}')  # DEBUG

    # Q5. Provide the numerical values for Calories (kcal), Fat (g), Protein (g), and Carbohydrates (g) for the entire dish, separated by commas.
    # Example answer: 200.0, 5.0, 20.0, 10.0
    macros = cur_img_data['nutritional_profile']
    macros_cleaned = {"calories_kcal": float(macros["calories_kcal"]), "fat_g": float(macros["fat_g"]), "protein_g": float(macros["protein_g"]), "carbs_g": float(macros["carbohydrate_g"])}
    macros_string = f"{macros_cleaned['calories_kcal']}, {macros_cleaned['fat_g']}, {macros_cleaned['protein_g']}, {macros_cleaned['carbs_g']}"
    qa_pairs.append({
            "question": "Provide the numerical values for Calories (kcal), Fat (g), Protein (g), and Carbohydrates (g) for the entire dish, separated by commas.",
            "answer": macros_string,
        })

    # Q6. Provide the numerical values for Total Weight (g), Iron (mg), Calcium (mg), and Vitamin C (mg), separated by commas.
    # Example answer: 500.0, 0.90, 160.0, 0.50
    micros_cleaned = {"total_weight_g": total_weight, "iron_mg": cur_img_data['Iron, Fe'], "calcium_mg": cur_img_data['Calcium, Ca'], "vitamin_C_mg": cur_img_data['Vitamin C, total ascorbic acid']}
    micros_string = f"{micros_cleaned['total_weight_g']}, {micros_cleaned['iron_mg']}, {micros_cleaned['calcium_mg']}, {micros_cleaned['vitamin_C_mg']}"
    qa_pairs.append({
            "question": "Provide the numerical values for Total Weight (g), Iron (mg), Calcium (mg), and Vitamin C (mg), separated by commas.",
            "answer": micros_string,
        })

    return qa_pairs

In [12]:
def generate_all_qa_pairs(in_path: str = '../data/train', out_path: str = '../data/train_demo', max_imgs: int = -1, outfile: str = 'my_qa_pairs.json'):
    from tqdm import tqdm
    split = in_path.split('/')[-1]
    csv_path = f'{in_path}/df_{split}_with_iron.csv'
    outfile_path = f'{out_path}/{outfile}'
    # Check if output file already exists
    if Path(outfile_path).exists():
        print(f"Output file {outfile} already exists. Please remove it first.")
        return
    # Create new output file
    with open(outfile_path, 'w') as file:
        file.write("[\n")  # Start an open list
        first_entry = True

        # Load input data
        df = pd.read_csv(csv_path)
        df['portion_size'] = df['portion_size'].apply(ast.literal_eval)
        df['nutritional_profile'] = df['nutritional_profile'].apply(ast.literal_eval)
        if max_imgs > 0:
            df = df.sample(n=max_imgs, random_state=42)
        
        print(f"Generating QA pairs for images in {in_path}...")
        for _, row in tqdm(df.iterrows(), total=len(df)):
            image_file = row.image_url.split('/')[-1]
            qa_pairs = generate_qa_pairs(row)
            qa_pairs = [{"question": qa["question"], "answer": qa["answer"], 
                        "image_file": f"{split}/{image_file}"} for qa in qa_pairs]
            # Add QA pairs to output file
            for qa in qa_pairs:
                if not first_entry:
                    file.write(",\n")
                json.dump(qa, file, indent=4)
                first_entry = False

        file.write("\n]\n")  # end list

## Train Demo

In [13]:
generate_all_qa_pairs(in_path='../data/train', out_path='../data/train_demo', max_imgs=2)

Generating QA pairs for images in ../data/train...


100%|██████████| 2/2 [00:00<00:00, 2706.00it/s]


## Val

In [14]:
generate_all_qa_pairs(in_path='../data/val', out_path='../data/val-grader', max_imgs=-1)

Generating QA pairs for images in ../data/val...


100%|██████████| 500/500 [00:00<00:00, 14733.30it/s]

Invalid weight! fried egg 1
Invalid weight! coconut  1 medium
Invalid weight! eggs 2
Invalid weight! eggs  3 large
Invalid weight! bananas  10 medium
Invalid weight! eggs 2


## Test

In [15]:
generate_all_qa_pairs(in_path='../data/test', out_path='../data/test-grader', max_imgs=-1)

Generating QA pairs for images in ../data/test...


100%|██████████| 500/500 [00:00<00:00, 17963.53it/s]

Invalid weight! egg tart  6 pieces
Invalid weight! eggs  20 pieces


## Train

In [16]:
generate_all_qa_pairs(in_path='../data/train', out_path='../data/train-grader', max_imgs=-1)

Generating QA pairs for images in ../data/train...


  0%|          | 0/4000 [00:00<?, ?it/s]

Invalid weight! bananas  4 medium (480g)
Invalid weight! eggs 2
Invalid weight! egg 2 large
Invalid weight! eggs 2
Invalid weight! eggs 3
Invalid weight! egg 1
Invalid weight! egg 3 large
Invalid weight! eggs 2
Invalid weight! eggs 4


 94%|█████████▎| 3747/4000 [00:00<00:00, 18754.48it/s]

Invalid weight! eggs 6
Invalid weight! eggs 2
Invalid weight! eggs 2
Invalid weight! pizza  1 slice (150g)
Invalid weight! egg 2 large
Invalid weight! eggs  6 large
Invalid weight! eggs 4
Invalid weight! eggs 4
Invalid weight! eggs 2
Invalid weight! eggs 3


100%|██████████| 4000/4000 [00:00<00:00, 18686.18it/s]

Invalid weight! egg 1
